# Генерация данных

In [1]:
import mlflow
import mlflow.sklearn

import numpy as np
import pandas as pd
import seaborn
import matplotlib.pyplot as plt 

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.datasets import make_classification

X, y = make_classification(n_samples= 3000, n_features=4, n_redundant=1, random_state=2042026, class_sep=0.3)
data = pd.DataFrame( data=np.hstack( [X, y.reshape(-1,1)], ), columns=["x1", "x2", "x3", "x4", "y"])

Train, Test = train_test_split(data, test_size=0.2, random_state=2042026)
X_train, y_train = Train.drop(columns=['y']).values, Train['y'].values
X_test, y_test = Test.drop(columns='y').values, Test['y'].values


Train.sample(10, random_state=0)

,x1,x2,x3,x4,y
275,-1.130790,0.684333,1.025889,-1.240076,0.0
858,1.478094,-0.466396,-1.304609,0.986051,1.0
824,2.341013,-0.025001,-1.872324,-1.823945,1.0
2611,1.574191,0.171113,-1.393324,1.118194,1.0
1554,-2.012680,0.272615,1.771971,-1.264493,0.0
632,-1.115015,1.009414,0.985225,-0.762693,0.0
264,-0.502651,0.353429,0.412222,0.213445,1.0
2016,-0.046488,-0.397830,0.056153,-0.295012,0.0
2991,-0.663717,-0.821356,0.530100,0.529963,1.0
1566,0.869704,-1.521754,-0.815949,1.423854,1.0


# Модели МО

In [23]:
mlflow.set_experiment("Synthetic_data_classification_02.04.2026_ver2")

2026/04/02 15:57:22 INFO mlflow.tracking.fluent: Experiment with name 'Synthetic_data_classification_02.04.2026_ver2' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/s/w/prep/computer science/ML/tools/mlflow/mlruns/3', creation_time=1775113042256, experiment_id='3', last_update_time=1775113042256, lifecycle_stage='active', name='Synthetic_data_classification_02.04.2026_ver2', tags={}, workspace='default'>

In [6]:
from sklearn.base import BaseEstimator


def fit_and_test(model: BaseEstimator, X_train:np.ndarray, X_test:np.ndarray, y_train:np.ndarray, y_test: np.ndarray):
    """Обучает model и печатает таблицы с метриками качества
    Возвращает accuracy_train, accuracy_test
    """
    model.fit(X_train, y_train)

    y_train_pred = model.predict( X_train )
    y_test_pred = model.predict( X_test )

    print("Train:")
    print(classification_report(y_train, y_train_pred))

    print("\n\nTest:")
    print(classification_report(y_test, y_test_pred))
    
    return accuracy_score(y_train, y_train_pred), accuracy_score(y_test, y_test_pred)

## LogReg

In [24]:
from sklearn.linear_model import LogisticRegression


# Создать новый эксперимент (run)
mlflow.start_run()


log_reg = LogisticRegression()
acc_train, acc_test = fit_and_test( log_reg, X_train, X_test, y_train, y_test)

# информация о данных
mlflow.log_params( {"train_size": len(y_train), "test_size": len(y_test) 
                    ,"class_balance": y_train.mean()        # баланс классов
                    ,"features_n": X_train.shape[1]         # количество признаков
                    ,"features": Train.columns
                    ,"outliers": ""
                    ,"scaling": ""
                    ,"missing_process": ""
                    ,"features_code": ""
                    } )

# информация о моделях
mlflow.log_params( {"model_type": "LogReg"} )
mlflow.log_metrics({"acc_train": acc_train, "acc_test": acc_test})
mlflow.sklearn.log_model( log_reg )

mlflow.end_run()

2026/04/02 15:57:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train:
              precision    recall  f1-score   support

         0.0       0.63      0.65      0.64      1197
         1.0       0.64      0.63      0.64      1203

    accuracy                           0.64      2400
   macro avg       0.64      0.64      0.64      2400
weighted avg       0.64      0.64      0.64      2400



Test:
              precision    recall  f1-score   support

         0.0       0.65      0.60      0.62       309
         1.0       0.60      0.66      0.63       291

    accuracy                           0.62       600
   macro avg       0.63      0.63      0.62       600
weighted avg       0.63      0.62      0.62       600



2026/04/02 15:57:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [15]:
knn.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [28]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=30, metric='manhattan')
acc_train, acc_test = fit_and_test( knn, X_train, X_test, y_train, y_test)

with mlflow.start_run():
    # информация о данных
    mlflow.log_params( {"train_size": len(y_train), "test_size": len(y_test) 
                        ,"class_balance": y_train.mean()        # баланс классов
                        ,"features_n": X_train.shape[1]         # количество признаков
                        ,"features": Train.columns
                        ,"outliers": ""
                        ,"scaling": ""
                        ,"missing_process": ""
                        ,"features_code": ""
                        } )

    # информация о моделях
    mlflow.log_params( {"model_type": "KNN"} )
    # mlflow.log_params( {"model_type": str(knn.__class__)} )
    mlflow.log_params( {"knn_k": knn.n_neighbors
                        ,"knn_weights": knn.weights
                        ,"knn_metric": knn.metric}) 
    mlflow.log_metrics({"acc_train": acc_train, "acc_test": acc_test})
    mlflow.sklearn.log_model( knn  )



2026/04/02 15:57:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train:
              precision    recall  f1-score   support

         0.0       0.66      0.81      0.73      1197
         1.0       0.75      0.58      0.66      1203

    accuracy                           0.70      2400
   macro avg       0.71      0.70      0.69      2400
weighted avg       0.71      0.70      0.69      2400



Test:
              precision    recall  f1-score   support

         0.0       0.62      0.75      0.68       309
         1.0       0.65      0.51      0.57       291

    accuracy                           0.63       600
   macro avg       0.64      0.63      0.62       600
weighted avg       0.64      0.63      0.63       600



2026/04/02 15:57:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


## Дерево Решений

In [48]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, criterion='entropy')
acc_train, acc_test = fit_and_test( tree, X_train, X_test, y_train, y_test)
with mlflow.start_run():
    mlflow.log_params( {"train_size": len(y_train), "test_size": len(y_test) 
                        ,"class_balance": y_train.mean()        # баланс классов
                        ,"features_n": X_train.shape[1]         # количество признаков
                        ,"features": Train.columns
                        ,"outliers": ""
                        ,"scaling": ""
                        ,"missing_process": ""
                        ,"features_code": ""
                        } )

    # информация о моделях
    mlflow.log_params( {"model_type": "decision tree"} )
    # mlflow.log_params( {"model_type": str(knn.__class__)} )
    mlflow.log_params( {"tree_max_depth": tree.max_depth
                        ,"tree_H": tree.criterion
                        }) 
    mlflow.log_metrics({"acc_train": acc_train, "acc_test": acc_test})
    mlflow.sklearn.log_model( knn  )

2026/04/02 17:22:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train:
              precision    recall  f1-score   support

         0.0       0.64      0.80      0.71      1197
         1.0       0.74      0.55      0.63      1203

    accuracy                           0.67      2400
   macro avg       0.69      0.67      0.67      2400
weighted avg       0.69      0.67      0.67      2400



Test:
              precision    recall  f1-score   support

         0.0       0.60      0.76      0.67       309
         1.0       0.65      0.47      0.54       291

    accuracy                           0.62       600
   macro avg       0.63      0.62      0.61       600
weighted avg       0.63      0.62      0.61       600



2026/04/02 17:22:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [31]:
tree.get_depth()

35

In [30]:
tree.get_params()

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [19]:
mlflow.search_runs()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.acc_test,metrics.acc_train,params.knn_weights,params.knn_k,...,params.features_n,params.features,params.test_size,params.train_size,params.scaling,params.class_balance,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.user
0,015064c216c549888a42ccf1f5028d62,2,FINISHED,/home/s/w/prep/computer science/ML/tools/mlflo...,2026-04-02 06:47:04.894000+00:00,2026-04-02 06:47:08.234000+00:00,0.610000,0.752500,uniform,5,...,4,"Index(['x1', 'x2', 'x3', 'x4', 'y'], dtype='ob...",600,2400,,0.50125,NOTEBOOK,lab2.ipynb,gregarious-koi-284,s
1,c78064fb0f784f3a95228a7868f46c93,2,FINISHED,/home/s/w/prep/computer science/ML/tools/mlflo...,2026-04-02 06:46:37.830000+00:00,2026-04-02 06:46:41.335000+00:00,0.645000,0.718750,uniform,10,...,4,"Index(['x1', 'x2', 'x3', 'x4', 'y'], dtype='ob...",600,2400,,0.50125,NOTEBOOK,lab2.ipynb,selective-shrike-215,s
2,9b61c126cc0044aeb2529817a85a4d95,2,FINISHED,/home/s/w/prep/computer science/ML/tools/mlflo...,2026-04-02 06:46:13.503000+00:00,2026-04-02 06:46:17.199000+00:00,0.598333,0.761250,uniform,5,...,4,"Index(['x1', 'x2', 'x3', 'x4', 'y'], dtype='ob...",600,2400,,0.50125,NOTEBOOK,lab2.ipynb,fun-colt-178,s
3,be0fb29d25dc45098081e70411098708,2,FAILED,/home/s/w/prep/computer science/ML/tools/mlflo...,2026-04-02 06:41:51.363000+00:00,2026-04-02 06:41:51.894000+00:00,0.598333,0.761250,None,None,...,4,"Index(['x1', 'x2', 'x3', 'x4', 'y'], dtype='ob...",600,2400,,0.50125,NOTEBOOK,lab2.ipynb,popular-foal-381,s
4,295ba3aa360a4edfb1eaa5aaa80b0be4,2,FINISHED,/home/s/w/prep/computer science/ML/tools/mlflo...,2026-04-02 06:39:39.586000+00:00,2026-04-02 06:39:44.820000+00:00,0.625000,0.637917,None,None,...,4,"Index(['x1', 'x2', 'x3', 'x4', 'y'], dtype='ob...",600,2400,,0.50125,NOTEBOOK,lab2.ipynb,glamorous-ant-938,s


In [20]:
mlflow.get_tracking_uri()

'sqlite:////home/s/w/prep/computer%20science/ML/tools/mlflow/mlflow.db'